# pypgo Mesh API Demo

This notebook covers the M1 mesh API surface:

- **MeshData** — `TriMeshData`, `TetMeshData`, `CubicMeshData` (NumPy-backed data containers)
- **MeshGeo** — `TriMeshGeo`, `TetMeshGeo`, `CubicMeshGeo` (geometry facades)
- **Conversion** — `to_mesh_data()` / `from_mesh_data()` bridge
- **ENuMaterial / MaterialSpec** — ENu material data carrier
- **VolumeMesh** — Vega volume mesh (accepts MeshData, rejects MeshGeo)
- **I/O** — `pypgo.mesh.read_obj/write_obj`, `pypgo.mesh.veg.read_veg/write_veg`

In [15]:
import numpy as np
import pypgo as pgo
from pypgo.mesh import (
    TriMeshData, TetMeshData, CubicMeshData,
    TriMeshGeo, TetMeshGeo, CubicMeshGeo,
    MeshDataType,
)
from pypgo.mesh.veg import MaterialSpec, VolumeMesh, VegFile
from pypgo.mesh.geo import TriMeshGeo, TetMeshGeo, CubicMeshGeo
from pypgo import io

## 1. Creating MeshData

`MeshData` objects own vertices and elements as plain data. They accept any array-like input (list, NumPy array, float32/float64, int32/int64).

In [16]:
# TriMeshData — triangle surface mesh (3 vertices per element)
tri_verts = [[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0]]
tri_faces = [[0, 1, 2], [1, 3, 2]]
tri_data = TriMeshData(tri_verts, tri_faces)
tri_data

In [17]:
# TetMeshData — tetrahedral volume mesh (4 vertices per element)
tet_verts = np.array([
    [0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1],
    [1, 0, 1], [1, 1, 0], [0, 1, 1], [1, 1, 1],
], dtype=np.float64)
tet_elems = np.array([
    [0, 1, 2, 3], [1, 4, 2, 3],
    [1, 5, 2, 4], [4, 5, 2, 6],
    [3, 4, 6, 2], [4, 5, 6, 7],
], dtype=np.int64)
tet_data = TetMeshData(tet_verts, tet_elems)
tet_data

In [18]:
# CubicMeshData — hexahedral volume mesh (8 vertices per element)
cubic_verts = np.array([
    [0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
    [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1],
], dtype=np.float64)
cubic_elems = [[0, 1, 2, 3, 4, 5, 6, 7]]
cubic_data = CubicMeshData(cubic_verts, cubic_elems)
cubic_data

## 2. MeshData Properties

All `MeshData` types expose the same interface: `num_vertices`, `num_elements`, `vertices`, `elements`, `mesh_type`, `element_vtx_id`.

In [19]:
print(f"TriMeshData:  {tri_data.num_vertices} vertices, {tri_data.num_elements} elements")
print(f"TetMeshData:  {tet_data.num_vertices} vertices, {tet_data.num_elements} elements")
print(f"CubicMeshData: {cubic_data.num_vertices} vertices, {cubic_data.num_elements} elements")

TriMeshData:  4 vertices, 2 elements
TetMeshData:  8 vertices, 6 elements
CubicMeshData: 8 vertices, 1 elements


In [20]:
# vertices returns (n, 3) float64, elements returns (m, K) int64
print("tri  vertices shape:", tri_data.vertices.shape, " dtype:", tri_data.vertices.dtype)
print("tri  elements shape:", tri_data.elements.shape, " dtype:", tri_data.elements.dtype)
print()
print("tet  vertices shape:", tet_data.vertices.shape)
print("tet  elements shape:", tet_data.elements.shape)
print()
print("cubic vertices shape:", cubic_data.vertices.shape)
print("cubic elements shape:", cubic_data.elements.shape)

tri  vertices shape: (4, 3)  dtype: float64
tri  elements shape: (2, 3)  dtype: int64

tet  vertices shape: (8, 3)
tet  elements shape: (6, 4)

cubic vertices shape: (8, 3)
cubic elements shape: (1, 8)


In [21]:
# mesh_type identifies the element kind
print("TriMeshData  type:", tri_data.mesh_type, "->", MeshDataType.Triangle == tri_data.mesh_type)
print("TetMeshData  type:", tet_data.mesh_type, "->", MeshDataType.Tet == tet_data.mesh_type)
print("CubicMeshData type:", cubic_data.mesh_type, "->", MeshDataType.Cubic == cubic_data.mesh_type)

TriMeshData  type: MeshDataType.Triangle -> True
TetMeshData  type: MeshDataType.Tet -> True
CubicMeshData type: MeshDataType.Cubic -> True


In [22]:
# element_vtx_id(element_index, local_vertex_index) -> global vertex index
print("tri  element 0, local vtx 0 -> global vtx", tri_data.element_vtx_id(0, 0))
print("tri  element 0, local vtx 2 -> global vtx", tri_data.element_vtx_id(0, 2))
print("tet  element 1, local vtx 3 -> global vtx", tet_data.element_vtx_id(1, 3))
print("cubic element 0, local vtx 7 -> global vtx", cubic_data.element_vtx_id(0, 7))

tri  element 0, local vtx 0 -> global vtx 0
tri  element 0, local vtx 2 -> global vtx 2
tet  element 1, local vtx 3 -> global vtx 3
cubic element 0, local vtx 7 -> global vtx 7


## 3. MeshGeo — Geometry Facades

`MeshGeo` objects provide typed accessors (`triangles`, `tets`, `cubes`, `tri_vtx_id`, etc.) and can be constructed directly from arrays or from `MeshData`.

In [23]:
# Direct construction from arrays
tri_geo = TriMeshGeo(tri_verts, tri_faces)
print(f"TriMeshGeo: {tri_geo.num_vertices} vertices, {tri_geo.num_triangles} triangles")
print("triangles:\n", tri_geo.triangles)
print("tri_vtx_id(0, 0):", tri_geo.tri_vtx_id(0, 0))

TriMeshGeo: 4 vertices, 2 triangles
triangles:
 [[0 1 2]
 [1 3 2]]
tri_vtx_id(0, 0): 0


In [24]:
tet_geo = TetMeshGeo(tet_verts, tet_elems)
print(f"TetMeshGeo: {tet_geo.num_vertices} vertices, {tet_geo.num_tets} tets")
print("tets:\n", tet_geo.tets)
print("tet_vtx_id(2, 1):", tet_geo.tet_vtx_id(2, 1))

TetMeshGeo: 8 vertices, 6 tets
tets:
 [[0 1 2 3]
 [1 4 2 3]
 [1 5 2 4]
 [4 5 2 6]
 [3 4 6 2]
 [4 5 6 7]]
tet_vtx_id(2, 1): 5


In [25]:
cubic_geo = CubicMeshGeo(cubic_verts, cubic_elems)
print(f"CubicMeshGeo: {cubic_geo.num_vertices} vertices, {cubic_geo.num_cubes} cubes")
print("cubes:\n", cubic_geo.cubes)
print("cube_vtx_id(0, 4):", cubic_geo.cube_vtx_id(0, 4))

CubicMeshGeo: 8 vertices, 1 cubes
cubes:
 [[0 1 2 3 4 5 6 7]]
cube_vtx_id(0, 4): 4


## 4. MeshData <-> MeshGeo Conversion

`MeshData` is the canonical intermediate representation. Convert with `to_mesh_data()` / `from_mesh_data()`.

In [26]:
# MeshData -> MeshGeo
geo_from_data = TetMeshGeo.from_mesh_data(tet_data)
print("Geo from MeshData:", geo_from_data)
print("Same tets?", np.array_equal(geo_from_data.tets, tet_data.elements))

Geo from MeshData: <pypgo.mesh.TetMeshGeo object at 0x1162734d0>
Same tets? True


In [27]:
# MeshGeo -> MeshData
data_from_geo = geo_from_data.to_mesh_data()
print("MeshData from Geo:", data_from_geo)
print("Same vertices?", np.array_equal(data_from_geo.vertices, tet_geo.vertices))
print("Same elements?", np.array_equal(data_from_geo.elements, tet_geo.tets))

MeshData from Geo: <pypgo.mesh.TetMeshData object at 0x1119b6190>
Same vertices? True
Same elements? True


In [28]:
# Roundtrip: MeshData -> Geo -> MeshData
tri_roundtrip = TriMeshGeo.from_mesh_data(tri_data).to_mesh_data()
print("Roundtrip ok:", np.array_equal(tri_data.vertices, tri_roundtrip.vertices)
      and np.array_equal(tri_data.elements, tri_roundtrip.elements))

Roundtrip ok: True


## 5. MaterialSpec

Lightweight ENu material value object.

In [29]:
mat = MaterialSpec(E=1e9, nu=0.45, density=1000.0)
print(mat)
print(f"E={mat.E}, nu={mat.nu}, density={mat.density}")

MaterialSpec(E=1000000000.0, nu=0.45, density=1000.0)
E=1000000000.0, nu=0.45, density=1000.0


## 6. VolumeMesh

Vega volume mesh constructed from `MeshData` + `MaterialSpec`. Accepts `TetMeshData` or `CubicMeshData`; **rejects** `MeshGeo` and `TriMeshData`.

In [30]:
# Construct from TetMeshData
vol_tet = VolumeMesh(tet_data, mat)
print(vol_tet)
print("mesh_type:", vol_tet.mesh_type)
print("material:", vol_tet.material)

VolumeMesh(type=MeshType.Tet, vertices=8, elements=6)
mesh_type: MeshType.Tet
material: MaterialSpec(E=1000000000.0, nu=0.45, density=1000.0)


In [31]:
# Construct from CubicMeshData
vol_cubic = VolumeMesh(cubic_data, mat)
print(vol_cubic)

VolumeMesh(type=MeshType.Cubic, vertices=8, elements=1)


In [32]:
# VolumeMesh.geometry / .mesh_data — lazy access back to MeshData
exported_data = vol_tet.mesh_data
print(type(exported_data).__name__)
print("Same elements?", np.array_equal(exported_data.elements, tet_data.elements))

TetMeshData
Same elements? True


In [33]:
# VolumeMesh rejects TriMeshData (surface mesh)
try:
    VolumeMesh(tri_data, mat)
except TypeError as e:
    print(f"TypeError (expected): {e}")

TypeError (expected): mesh_data must be a TetMeshData or CubicMeshData, got TriMeshData


In [34]:
# VolumeMesh rejects MeshGeo (must use MeshData)
try:
    VolumeMesh(tet_geo, mat)
except TypeError as e:
    print(f"TypeError (expected): {e}")

TypeError (expected): mesh_data must be a TetMeshData or CubicMeshData, got TetMeshGeo


## 7. I/O — Reading and Writing Mesh Files

All I/O functions accept and return `MeshData` (not `MeshGeo`).

In [35]:
# Write then read back an OBJ (TriMeshData)
import tempfile, os
tmpdir = tempfile.mkdtemp()

obj_path = os.path.join(tmpdir, "test.obj")
pgo.mesh.write_obj(obj_path, tri_data)
tri_loaded = pgo.mesh.read_obj(obj_path)
print(f"OBJ roundtrip: {tri_loaded.num_vertices} vertices, {tri_loaded.num_elements} elements")
print("Vertices match:", np.allclose(tri_data.vertices, tri_loaded.vertices))
print("Elements match:", np.array_equal(tri_data.elements, tri_loaded.elements))

OBJ roundtrip: 4 vertices, 2 elements
Vertices match: True
Elements match: True
Saved mesh (#v: 4, #t: 2) to /var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/tmpzvs3clpk/test.obj.


In [36]:
# Write then read back a VEG (TetMeshData + MaterialSpec)
veg_path = os.path.join(tmpdir, "test.veg")
pgo.mesh.veg.write_veg(veg_path, VegFile.from_single_material(tet_data, mat))
veg = pgo.mesh.veg.read_veg(veg_path)
veg_data, veg_mat = veg.mesh_data, veg.first_material()
print(f"VEG roundtrip: {veg_data.num_vertices} vertices, {veg_data.num_elements} elements")
print(f"Material: E={veg_mat.E}, nu={veg_mat.nu}")
print("Vertices match:", np.allclose(tet_data.vertices, veg_data.vertices))

VEG roundtrip: 8 vertices, 6 elementsOpening file /var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/tmpzvs3clpk/test.veg.

Material: E=1000000000.0, nu=0.45
Vertices match: True


In [37]:
# VolumeMesh.load — load .veg directly into a simulation mesh
vol_from_file = VolumeMesh.load(veg_path)
print(vol_from_file)
print("Material from loaded VolumeMesh:", vol_from_file.material)

VolumeMesh(type=MeshType.Tet, vertices=8, elements=6)
Material from loaded VolumeMesh: MaterialSpec(E=1000000000.0, nu=0.45, density=1000.0)
Opening file /var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/tmpzvs3clpk/test.veg.


In [38]:
# I/O rejects MeshGeo — only MeshData is accepted
try:
    pgo.mesh.write_obj(os.path.join(tmpdir, "bad.obj"), tri_geo)
except TypeError as e:
    print(f"TypeError (expected): {e}")

try:
    pgo.mesh.veg.write_veg(os.path.join(tmpdir, "bad.veg"), VegFile.from_single_material(tet_geo, mat))
except TypeError as e:
    print(f"TypeError (expected): {e}")

TypeError (expected): surface_data must be a TriMeshData, got TriMeshGeo
TypeError (expected): mesh_data must be a TetMeshData or CubicMeshData, got TetMeshGeo


In [39]:
# Cleanup
import shutil
shutil.rmtree(tmpdir)

## 8. Input Flexibility

`MeshData` and `MeshGeo` constructors accept lists, float32, int32, and mixed inputs. Everything is normalized to float64/int64 internally.

In [40]:
# float32 vertices, int32 elements — both accepted
v32 = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
e32 = np.array([[0, 1, 2, 3]], dtype=np.int32)
td = TetMeshData(v32, e32)
print("vertices dtype:", td.vertices.dtype, " elements dtype:", td.elements.dtype)

vertices dtype: float64  elements dtype: int64


In [41]:
# Plain lists also work
td2 = TetMeshData([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]],
                  [[0, 1, 2, 3]])
print(td2)

In [42]:
# Shape validation — wrong element width is rejected
try:
    TetMeshData(tet_verts, [[0, 1, 2]])  # 3 columns, need 4
except (ValueError, RuntimeError) as e:
    print(f"Error (expected): {e}")

Error (expected): elements must be a 2D array with 4 columns, got shape (1, 3)


## Summary

| Concept | Types | Role |
|---------|-------|------|
| **MeshData** | `TriMeshData`, `TetMeshData`, `CubicMeshData` | Canonical data container, I/O boundary |
| **MeshGeo** | `TriMeshGeo`, `TetMeshGeo`, `CubicMeshGeo` | Typed geometry facade with named accessors |
| **Conversion** | `.to_mesh_data()`, `.from_mesh_data()` | Explicit bridge, no implicit casting |
| **Material** | `MaterialSpec` | ENu material parameters |
| **VolumeMesh** | `pypgo.mesh.veg.VolumeMesh` | Vega volume mesh (MeshData + material/regions) |
| **I/O** | `pypgo.mesh.read_obj/write_obj`, `pypgo.mesh.veg.read_veg/write_veg`, `VolumeMesh.load` | Domain-scoped file I/O at MeshData boundary |